<a href="https://colab.research.google.com/github/MEpperley/MLP_Implementation_Code/blob/main/Recurrent_Neural_Networks_Sequence_SHAKESPEARE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================
# IMPORT LIBRARIES
# ================================

import torch  # PyTorch core library for tensor operations and deep learning
from torch import nn  # Neural network modules (RNN, Linear layers, etc.)
import random  # Used for random sampling of training sequences
import requests  # Used to download Shakespeare dataset from the internet


print("Complete: Libraries imported successfully.")  # Confirm imports worked

Complete: Libraries imported successfully.


In [ ]:
# ================================
# LOAD SHAKESPEARE DATASET
# ================================

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# URL pointing to small Shakespeare corpus used for character-level language modeling

text = requests.get(url).text  # Download dataset and convert response to plain text string

print("Dataset loaded. Total characters:", len(text))  # Show dataset size

Dataset loaded. Total characters: 1115394


In [ ]:
# ================================
# BUILD CHARACTER VOCABULARY
# ================================

chars = sorted(list(set(text)))
# Extract all unique characters in dataset and sort them

vocab_size = len(chars)
# Total number of unique characters in vocabulary

char_to_idx = {ch: i for i, ch in enumerate(chars)}
# Map each character → unique integer index

idx_to_char = {i: ch for i, ch in enumerate(chars)}
# Reverse mapping: integer index → character


print("Vocabulary size:", vocab_size)  # Print vocabulary size

Vocabulary size: 65


In [ ]:
# ================================
# ENCODE TEXT INTO INTEGERS
# ================================

data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
# Convert full text into sequence of integer token IDs

In [ ]:
# ================================
# DEFINE RNN MODEL
# ================================

class RNNModel(nn.Module):  # Define neural network class

    def __init__(self, vocab_size, num_hiddens):
        super().__init__()  # Initialize parent PyTorch module

        self.num_hiddens = num_hiddens  # Store hidden state size

        self.vocab_size = vocab_size  # Store vocabulary size

        self.rnn = nn.RNN(
            input_size=vocab_size,  # One-hot input size equals vocab size
            hidden_size=num_hiddens  # Hidden state dimension
        )  # RNN layer processes sequence step-by-step

        self.linear = nn.Linear(
            num_hiddens,  # Input: hidden state
            vocab_size  # Output: probability over characters
        )  # Fully connected layer for prediction


    def forward(self, inputs, state):

        X = nn.functional.one_hot(inputs.T, self.vocab_size).float()
        # Convert input indices → one-hot vectors
        # Transpose so time dimension comes first (RNN requirement)

        Y, state = self.rnn(X, state)
        # Pass sequence through RNN → get hidden states + updated state

        output = self.linear(Y.reshape(-1, Y.shape[-1]))
        # Flatten all time steps and apply linear projection to vocab space

        return output, state  # Return logits and updated hidden state


    def begin_state(self, batch_size):
        return torch.zeros((1, batch_size, self.num_hiddens))
        # Initialize hidden state as zeros (1 layer, batch_size, hidden size)


print("Complete: RNN model defined.")

Complete: RNN model defined.


In [ ]:
# ================================
# HYPERPARAMETERS
# ================================

num_hiddens = 256  # Size of hidden state vector (model capacity)
batch_size = 32  # Number of parallel sequences per batch
num_steps = 50  # Length of each training sequence
learning_rate = 0.01  # Step size for gradient descent
num_epochs = 10  # Number of full training passes over dataset


print("Complete: Hyperparameters set.")

Complete: Hyperparameters set.


In [ ]:
# ================================
# BATCH GENERATION FUNCTION
# ================================

def get_batch(data, batch_size, num_steps):

    num_tokens = (len(data) // batch_size) * batch_size
    # Trim data so it can be evenly divided across batch dimension

    data = data[:num_tokens]
    # Remove extra leftover tokens

    data = data.reshape(batch_size, -1)
    # Reshape into batch_size continuous streams

    start = random.randint(0, data.shape[1] - num_steps - 2)
    # Random starting point for sequence window

    X = data[:, start:start + num_steps]
    # Input sequence (current characters)

    Y = data[:, start + 1:start + num_steps + 1]
    # Target sequence (next-character prediction)

    return X, Y  # Return input-target pair

In [ ]:
# ================================
# INITIALIZE MODEL
# ================================

model = RNNModel(vocab_size, num_hiddens)
# Create RNN model instance


print("Complete: Model initialized.")

Complete: Model initialized.


In [ ]:
# ================================
# LOSS + OPTIMIZER
# ================================

loss_fn = nn.CrossEntropyLoss()
# Loss function for multi-class character prediction

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# Stochastic gradient descent optimizer


print("Complete: Loss and optimizer ready.")

Complete: Loss and optimizer ready.


In [ ]:
# ================================
# GRADIENT CLIPPING
# ================================

def grad_clipping(model, theta):

    params = [p for p in model.parameters() if p.requires_grad]
    # Collect trainable parameters

    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    # Compute global gradient norm

    if norm > theta:
        # If gradients too large (exploding gradients)

        for p in params:
            p.grad[:] *= theta / norm
            # Rescale gradients to stabilize training


print("Complete: Gradient clipping ready.")

Complete: Gradient clipping ready.


In [ ]:
# ================================
# TRAINING LOOP
# ================================

for epoch in range(num_epochs):  # Loop over dataset multiple times

    state = model.begin_state(batch_size)
    # Reset hidden state at start of each epoch

    total_loss = 0  # Track total loss

    for i in range(len(data) // (batch_size * num_steps)):
        # Iterate over dataset in chunks

        X, Y = get_batch(data, batch_size, num_steps)
        # Sample batch of sequences

        optimizer.zero_grad()
        # Reset gradients before backpropagation

        state = state.detach()
        # Detach hidden state from previous computation graph
        # to prevent backpropagating through the entire training history

        outputs, state = model(X, state)
        # Forward pass through RNN

        y = Y.T.reshape(-1)
        # Flatten targets for loss computation

        loss = loss_fn(outputs, y)
        # Compute cross-entropy loss

        loss.backward()
        # Backpropagation through time

        grad_clipping(model, 1.0)
        # Prevent exploding gradients

        optimizer.step()
        # Update model parameters

        total_loss += loss.item()
        # Accumulate loss for reporting

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")
    # Print progress after each epoch


print("Complete: Training finished.")

Epoch 1, Loss: 2422.9756
Epoch 2, Loss: 2301.7344
Epoch 3, Loss: 2295.2207
Epoch 4, Loss: 2282.4756
Epoch 5, Loss: 2261.6912
Epoch 6, Loss: 2230.5379
Epoch 7, Loss: 2185.9216
Epoch 8, Loss: 2130.7613
Epoch 9, Loss: 2083.6797
Epoch 10, Loss: 2044.1715
Complete: Training finished.


In [ ]:
# ================================
# TEXT GENERATION FUNCTION
# ================================

def predict(prefix, num_chars, model):

    state = model.begin_state(batch_size=1)
    # Initialize hidden state for generation

    outputs = [char_to_idx[c] for c in prefix]
    # Convert prefix text → integer tokens

    for idx in outputs:
        X = torch.tensor([[idx]])
        # Feed each prefix character into model
        # Wrap forward pass in torch.no_grad() to disable gradient
        # tracking during inference, avoids memory waste and errors when
        # the model is in eval-like usage outside of a training context

        with torch.no_grad():
            _, state = model(X, state)
            # Update hidden state (no output used yet)

    for _ in range(num_chars):
        # Generate new characters step-by-step

        X = torch.tensor([[outputs[-1]]])
        # Use last predicted character as input

        with torch.no_grad():
            # no_grad() required during generation
            # to prevent RuntimeError from gradient accumulation on leaf
            # tensors that are not part of any training graph

            y, state = model(X, state)
            # Get next-character prediction

        next_char = y.argmax(dim=1).item()
        # Pick most likely character (greedy decoding)

        outputs.append(next_char)
        # Append predicted character

    return "".join([idx_to_char[i] for i in outputs])
    # Convert indices back into readable text


print("Complete: Text generation ready.")

Complete: Text generation ready.


In [ ]:
# ================================
# GENERATE SAMPLE OUTPUT
# ================================

sample = predict("To be or not to be", 200, model)
# Generate 200 characters starting from prefix

print("\nGenerated Text:\n")
print(sample)
# Display generated Shakespeare-like text

print("\nComplete: Done.")


Generated Text:

To be or not to be the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the

Complete: Done.
